In [149]:
import os, sys
import polars as pl


In [150]:
script_path = os.getcwd()
project_path = os.path.join(script_path, '..', '..', '..')
data_dir = os.path.join(project_path, 'data', 'forms')
raw_data_dir = os.path.join(data_dir, 'raw_data')
processed_data_dir = os.path.join(data_dir, 'processed_data')

raw_data_filenames = [f for f in os.listdir(raw_data_dir) if 'forms' in f]

In [151]:
raw_data = {f.split('.')[0]: pl.read_csv(os.path.join(raw_data_dir, f)) for f in raw_data_filenames}


---

In [152]:
# Nomenclatura:
# tc = test conocimiento
# ta = test autoeficacia
# tcc = test carga congnitiva

# Global Variables
TC_TIPOS = ['retencion', 'transferencia']
PERIODOS = ['pre', 'post']
MAX_PUNTUACION_TC = 10
MAX_PUNTUACION_TC_TIPO = 5
MAX_PUNTUACION_TA = 7
MAX_PUNTUACION_TA = 10
TC_RESPUESTAS_CORRECTAS = {

    'pre': {

        'retencion': [
            'Describir y resumir un conjunto de datos recogidos sin sacar conclusiones sobre una población más amplia.',
            'Coche',
            6,
            'La distancia media aproximada entre cada dato concreto y la media del conjunto.',
            'Para analizar y comparar la relación entre dos variables categóricas simultáneamente.'
        ],

        'transferencia': [
            'La mediana, porque es resistente a los valores extremos y representa mejor al grupo mayoritario.',
            'Que el 50% de los alumnos sacó notas entre 50 y 80',
            'No, porque la estadística descriptiva sólo resume los datos recogidos (los 20 pacientes) sin sacar conclusiones de la población total.',
            'El Grupo B, porque sus valores están más dispersos y alejados de su media que los del Grupo A.',
            'Gráfico circular / tarta (Pie chart)'
        ]
    },

    'post': {

        'retencion': [
            'Resumir y describir los tiempos de esos 10 atletas específicamente.',
            'Rojo.',
            '25 m2 ambos.',
            'El grado medio de dispersión o separación de los valores alrededor de la media.',
            'Mostrar con qué frecuencia aparece cada valor distinto en una sola variable.'
        ],

        'transferencia': [
            'Porque la media será mucho más alta que el precio real de la mayoría de las casas debido al valor atípico.',
            'Que el 50% de los empleados tiene sueldos comprendidos entre 1500 € y 2500 €.',
            'No, la estadística descriptiva sólo describe el grupo analizado (Clase A); extenderlo a la Clase B sería inferencia.',
            'Ruta 2 tiene mayor desviación; es menos predecible y los datos están más lejos de la media.',
            'Un gráfico / diagrama de barras'
        ]
    }
}

In [153]:
def get_cols_tc(df):

    return {
        tc_tipo: [
            c for c in df.columns if 'conocimientos' in c and tc_tipo in c
        ] 
        for tc_tipo in TC_TIPOS
    }

def get_exprs_tc_tipos(cols_tc):

    return {
        periodo: {
            tc_tipo: [
                pl.col(c) == TC_RESPUESTAS_CORRECTAS[periodo][tc_tipo][i] 
                for i, c in enumerate(cols_tc[tc_tipo])
            ]
            for tc_tipo in TC_TIPOS
        } 
        for periodo in PERIODOS
    }

def get_cols_ta(df):

    return [
        c for c in df.columns if 'autoeficacia' in c
    ] 

def get_exprs_ta(cols_ta):

    return [
        pl.col(c) / MAX_PUNTUACION_TA
        for c in cols_ta
    ]

def get_cols_tcc_rel(df):

    return [
        c for c in df.columns if 'carga' in c and 'relevante' in c
    ] 

def get_exprs_tcc_rel(cols_tcc):

    return [
        pl.col(c) / MAX_PUNTUACION_TC
        for c in cols_tcc
    ]

def get_cols_tcc_int(df):

    return [
        c for c in df.columns if 'carga' in c and 'intrinseca' in c
    ] 

def get_exprs_tcc_int(cols_tcc):

    return [
        pl.col(c) / MAX_PUNTUACION_TC
        for c in cols_tcc
    ]


def get_cols_tcc_ext(df):

    return [
        c for c in df.columns if 'carga' in c and 'extrinseca' in c
    ] 

def get_exprs_tcc_ext(cols_tcc):

    return [
        pl.col(c) / MAX_PUNTUACION_TC
        for c in cols_tcc
    ]

In [156]:
def process_data(raw_data_dir, processed_data_dir):

    raw_data_filenames = [f for f in os.listdir(raw_data_dir) if 'forms' in f]

    raw_data = {f: pl.read_csv(os.path.join(raw_data_dir, f)) for f in raw_data_filenames}

    # TODO: eliminar cuando la función esté construida: 
    processed_data = {}

    for raw_filename, df_raw in raw_data.items():

        print(raw_filename)

        df = df_raw.clone()
        
        df = df.rename(
            {c: c.lower().replace('retención', 'retencion').replace('extrínseca', 'extrinseca').replace('intrínseca', 'intrinseca') for c in df.columns}
        ).rename({'identificador': 'id'})

        cols_tc = get_cols_tc(df)
        cols_ta = get_cols_ta(df)
        cols_tcc_rel = get_cols_tcc_rel(df)
        cols_tcc_int = get_cols_tcc_int(df)
        cols_tcc_ext = get_cols_tcc_ext(df)
        exprs_tc_tipos = get_exprs_tc_tipos(cols_tc)
        exprs_ta = get_exprs_ta(cols_ta)
        exprs_tcc_rel = get_exprs_tcc_rel(cols_tcc_rel)
        exprs_tcc_int = get_exprs_tcc_int(cols_tcc_int)
        exprs_tcc_ext = get_exprs_tcc_ext(cols_tcc_ext)
        exprs_tc_tipos = exprs_tc_tipos['pre'] if 'pre' in raw_filename else exprs_tc_tipos['post']

        df = (
                df\
                .with_columns(
                    [
                        pl.lit('pre').alias('periodo') if 'pre' in raw_filename else pl.lit('post').alias('periodo'),
                        (pl.col('puntuación').str.splitn(" / ", 2).struct.field("field_0").cast(pl.Int64) / MAX_PUNTUACION_TC).alias('puntuacion_tc'),
                        (pl.sum_horizontal(exprs_ta) / len(cols_ta)).round(2).alias('puntuacion_ta'),
                        (pl.sum_horizontal(exprs_tcc_rel) / len(cols_tcc_rel)).round(2).alias('puntuacion_tcc_rel') if 'post' in raw_filename else pl.lit(None).alias('puntuacion_tcc_rel'),
                        (pl.sum_horizontal(exprs_tcc_int) / len(cols_tcc_int)).round(2).alias('puntuacion_tcc_int') if 'post' in raw_filename else pl.lit(None).alias('puntuacion_tcc_int'),
                        (pl.sum_horizontal(exprs_tcc_ext) / len(cols_tcc_ext)).round(2).alias('puntuacion_tcc_ext') if 'post' in raw_filename else pl.lit(None).alias('puntuacion_tcc_ext')
                    ] + 
                    [
                        (pl.sum_horizontal(exprs_tc_tipos[tc_tipo]) / MAX_PUNTUACION_TC_TIPO).alias(f'puntuacion_tc_{tc_tipo}') for tc_tipo in TC_TIPOS
                    ]
                )
            )
                
        processed_data_path = os.path.join(processed_data_dir, f'processed_{raw_filename}')
        #df.write_csv(processed_data_path)

        # TODO: eliminar cuando la función esté construida: 
        processed_data[f'processed_{raw_filename}'] = df

    return processed_data

In [157]:
processed_data = process_data(raw_data_dir=raw_data_dir, processed_data_dir=processed_data_dir)

respuestas_forms_pre_RamiroMaeztu_20260218.csv
respuestas_forms_post_RamiroMaeztu_20260218.csv


In [158]:
processed_data['processed_respuestas_forms_pre_RamiroMaeztu_20260218.csv']

marca temporal,puntuación,id,[autoeficacia inicial] estoy seguro de que puedo entender los conceptos básicos sobre estadística descriptiva.,[autoeficacia inicial] confío en que puedo entender los conceptos más complejos sobre estadística descriptiva.,[conocimientos previos] [retencion] ¿cuál es el objetivo principal de la estadística descriptiva?,"[conocimientos previos] [transferencia] imagina una pequeña startup con 5 empleados. cuatro de ellos ganan 1.000 € al mes, y el fundador gana 20.000 € al mes. si quieres dar una imagen realista del salario """"típico"""" de un trabajador a un nuevo candidato, ¿qué medida de tendencia central deberías usar y por qué?","[conocimientos previos] [retencion] si realizamos una encuesta a 24 alumnos del colegio sobre cuál es la forma de transporte a través de la cual vienen al colegio, y se obtienen los siguientes resultados: 5 personas en bicicleta, 12 en coche, 5 a pie y 2 en autobús. ¿cuál es la moda?","[conocimientos previos] [retencion] se tiene el siguiente conjunto de notas de 4 alumnos de cierta asignatura: 8, 2, 4, 10. ¿cuál es la mediana?",[conocimientos previos] [retencion] ¿qué nos indica conceptualmente la desviación estándar (o típica)?,[conocimientos previos] [retencion] ¿para qué sirve específicamente una tabla de contingencia (cross tab)?,[conocimientos previos] [transferencia] un informe escolar indica que el rango intercuartílico (iqr) de las notas de una clase va de 50 a 80 puntos. ¿qué podemos deducir?,"[conocimientos previos] [transferencia] un hospital realiza un estudio descriptivo sobre la presión arterial de 20 pacientes ingresados en planta y calcula su media. basándonos en la distinción entre estadística descriptiva e inferencial, ¿es correcto afirmar que """"la presión media de todos los pacientes del hospital es esa""""?","[conocimientos previos] [transferencia] tienes dos grupos de alumnos en una misma asignatura. grupo a: puntuaciones [8, 9, 10, 8, 9]. grupo b: puntuaciones [0, 10, 0, 10, 5]. la puntuación media del grupo a es 8.8, la del b es 5. sin hacer cálculos complejos, ¿qué grupo tendrá una desviación estándar mayor?","[conocimientos previos] [transferencia] si el departamento de rrhh (recursos humanos) quiere visualizar qué porcentaje del total de empleados pertenece a cada departamento (ventas, it, marketing), ¿qué gráfico sería el más directo para ver """"partes de un todo""""?",[autoeficacia inicial] creo que puedo hacer un trabajo excelente en la prueba de conocimientos sobre estadística descriptiva.,[autoeficacia inicial] creo que podré utilizar lo que aprenda sobre estadística descriptiva en actividades futuras.,periodo,puntuacion_tc,puntuacion_ta,puntuacion_tcc_rel,puntuacion_tcc_int,puntuacion_tcc_ext,puntuacion_tc_retencion,puntuacion_tc_transferencia
str,str,str,i64,i64,str,str,str,i64,str,str,str,str,str,str,i64,i64,str,f64,f64,null,null,null,f64,f64
"""18/02/2026 9:03:26""","""8 / 10""","""RamiroMaeztu-9jg""",6,5,"""Describir y resumir un conjunt…","""El rango, porque muestra la di…","""Coche""",6,"""La distancia media aproximada …","""Para analizar y comparar la re…","""Que nadie sacó más de 80 punto…","""No, porque la estadística desc…","""El Grupo B, porque sus valores…","""Gráfico circular / tarta (Pie …",5,4,"""pre""",0.8,0.5,null,null,null,1.0,0.6
"""18/02/2026 9:03:39""","""8 / 10""","""RamiroMaeztu-unc""",5,5,"""Describir y resumir un conjunt…","""El rango, porque muestra la di…","""Coche""",6,"""La distancia media aproximada …","""Para analizar y comparar la re…","""Que nadie sacó más de 80 punto…","""No, porque la estadística desc…","""El Grupo B, porque sus valores…","""Gráfico circular / tarta (Pie …",4,5,"""pre""",0.8,0.48,null,null,null,1.0,0.6
"""18/02/2026 9:04:29""","""8 / 10""","""RamiroMaeztu-wr6""",6,1,"""Predecir el comportamiento fut…","""La mediana, porque es resisten…","""Coche""",6,"""La distancia media aproximada …","""Para analizar y comparar la re…","""Que el 50% de los alumnos sacó…","""No, a menos que utilicem

In [159]:
processed_data['processed_respuestas_forms_post_RamiroMaeztu_20260218.csv']

marca temporal,puntuación,id,"[conocimientos post] [retencion] según el video, si analizamos los tiempos de carrera de 10 atletas de un club, la estadística descriptiva nos permite:","[conocimientos post] [transferencia] en un barrio, casi todas las casas cuestan alrededor de 100.000 €, pero hay una mansión que cuesta 5.000.000 €. si quieres saber el precio de una casa """"estándar"""" en ese barrio para comprar, ¿por qué la media no sería una buena referencia?",[carga cognitiva post] [carga relevante] la actividad me ha ayudado mucho a entender el contenido del video.,[autoeficacia final] estoy seguro de que he logrado entender los conceptos básicos presentados en el video.,"[conocimientos post] [retencion] en una encuesta sobre colores favoritos se obtuvieron los siguientes resultados: 3 personas eligieron azul, 3 eligieron verde, 10 eligieron rojo y 4 eligieron amarillo. ¿cuál es la moda de este conjunto?","[conocimientos post] [retencion] tenemos dos edificios de 4 alturas, cuyas plantas tienen los siguientes m2: edificio a: 20 m2, 10 m2, 40 m2, 30 m2. edificio b: 7.5 m2, 30 m2, 20 m2, 90 m2. ¿cuáles son las medianas de los 2 edificios?","[conocimientos post] [retencion] según lo explicado en el video, ¿qué característica fundamental de los datos describe la desviación estándar?",[conocimientos post] [retencion] ¿cuál es la función principal de una tabla de frecuencias simple descrita en el video?,"[conocimientos post] [transferencia] en un análisis de salarios, el primer cuartil (q1) es 1500 € y el tercer cuartil (q3) es 2500 €. ¿qué información nos da el rango intercuartílico sobre los empleados?","[conocimientos post] [transferencia] un profesor calcula la nota media de los 30 alumnos de su clase de matemáticas (clase a). ¿puede usar ese único dato descriptivo para afirmar que """"los alumnos de la clase b tienen el mismo rendimiento""""?","[conocimientos post] [transferencia] imagina dos rutas de autobús al trabajo. ruta 1: tarda siempre 30 minutos (30, 30, 30, 30). ruta 2: a veces tarda 10 minutos y a veces 50 minutos (10, 50, 10, 50). ambas tienen una media de 30 minutos. ¿qué ruta tiene mayor desviación estándar y qué implica eso para tu puntualidad?",[conocimientos post] [transferencia] si quieres comparar la altura media de hombres vs mujeres por grupo de edad ¿qué tipo de gráficos mencionado en el video es adecuado para este análisis?,[carga cognitiva post] [carga relevante] la forma de trabajar ha facilitado la comprensión de los conceptos.,[carga cognitiva post] [carga relevante] he podido centrar mi esfuerzo mental en aprender las ideas principales.,[carga cognitiva post] [carga extrinseca] la forma de presentar la información hacía difícil aprender.,[carga cognitiva post] [carga extrinseca] he tenido que esforzarme mucho para manejar la actividad.,[carga cognitiva post] [carga extrinseca] interactuar con el entorno de aprendizaje ha sido confuso.,[carga cognitiva post] [carga intrinseca] los temas tratados en el video eran complejos.,[carga cognitiva post] [carga intrinseca] los conceptos explicados eran difíciles de entender.,[carga cognitiva post] [carga intrinseca] el contenido del video presentaba una dificultad elevada.,[autoeficacia final] confío en que he comprendido los conceptos complejos enseñados en el video.,[autoeficacia final] creo que he realizado un trabajo excelente en la prueba de conocimientos sobre estadística descriptiva.,[autoeficacia final] creo que podré utilizar lo que he aprendido sobre estadística descriptiva en actividades futuras.,periodo,puntuacion_tc,puntuacion_ta,puntuacion_tcc_rel,puntuacion_tcc_int,puntuacion_tcc_ext,puntuacion_tc_retencion,puntuacion_tc_transferencia
str,str,str,str,str,i64,i64,str,str,str,str,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,f64,f64,f64,f64,f64,f64,f64
"""18/02/2026 9:43:17""","""10 / 10""","""RamiroMaeztu-ks0""","""Resumir y describir los tiempo…","""Porque la media será mucho más…",8,5,"""Rojo.""","""25 m2 ambos.""","""El gr

---